# Beispiele: Ensembles

Random Forest (Bagging) und Gradient Boosting. Klassifikation mit `load_breast_cancer`, Regression mit `load_diabetes`.

- **Herkunft:** selbst erstellt für die Lernapp, kein HSLU-Kursmaterial. Die Daten sind in scikit-learn eingebaut oder werden erzeugt.
- **Ausführen:** von oben nach unten, das Notebook hängt von keinem anderen ab. Lokal in der Lernapp oder auf jedem anderen PC im Browser: [In Colab öffnen](https://colab.research.google.com/github/rotscher-baumi/lernapp-cas-ml/blob/main/notebooks/beispiele/03_Ensembles.ipynb) (beim ersten Mal mit GitHub anmelden und «Include private repos» anhaken).
- **Navigation:** Das Inhaltsverzeichnis links in JupyterLab springt zu jedem Abschnitt. Zurück zur Übersicht: Lernapp, Kapitel «Übersicht».

In [ ]:
# Colab has scikit-learn, pandas and matplotlib; umap and imblearn are installed there on demand
import importlib.util, subprocess, sys
fehlend = [pip for mod, pip in [("umap", "umap-learn"), ("imblearn", "imbalanced-learn")] if importlib.util.find_spec(mod) is None]
if fehlend and "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *fehlend], check=True)
elif fehlend:
    print("Für einzelne Beispiele fehlt: pip install " + " ".join(fehlend))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import (load_diabetes, load_breast_cancer, load_wine, load_digits,
                              make_moons, make_blobs, make_classification)
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, accuracy_score,
                             precision_score, recall_score, f1_score, confusion_matrix,
                             classification_report, roc_auc_score, silhouette_score,
                             adjusted_rand_score)

SEED = 42
plt.rcParams["figure.figsize"] = (6, 3.5)
pd.set_option("display.precision", 3)

## Daten

In [ ]:
from sklearn.tree import DecisionTreeClassifier

Xc, yc = load_breast_cancer(return_X_y=True, as_frame=True)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(Xc, yc, test_size=0.25, stratify=yc, random_state=SEED)
X, y = load_diabetes(return_X_y=True, as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

## Random Forest (Bagging)

Ein einzelner Baum gegen einen Wald aus 300 Bäumen, beide mit 5-facher Cross-Validation. Danach: Welche Merkmale nutzt der Wald am meisten?

In [ ]:
from sklearn.ensemble import RandomForestClassifier

baum_cv = cross_val_score(DecisionTreeClassifier(random_state=SEED), Xc_train, yc_train, cv=5)
wald = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1)
wald_cv = cross_val_score(wald, Xc_train, yc_train, cv=5)
print(f"Einzelbaum:    {baum_cv.mean():.3f} ± {baum_cv.std():.3f}")
print(f"Random Forest: {wald_cv.mean():.3f} ± {wald_cv.std():.3f}")

wald.fit(Xc_train, yc_train)
pd.Series(wald.feature_importances_, index=Xc.columns).sort_values(ascending=False).head(5)

**Deutung:** Der Wald ist genauer und schwankt weniger zwischen den Folds (kleinere Standardabweichung): Bagging senkt die Variance.

## Gradient Boosting

Lernrate und Anzahl Runden hängen zusammen: Eine kleine Lernrate braucht mehr Runden. Beispiel als Regression auf den Diabetes-Daten, verglichen mit linearer Regression und Random Forest.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor

zeilen = []
for lr in [0.01, 0.1, 0.5]:
    for n in [50, 200, 800]:
        gb = GradientBoostingRegressor(learning_rate=lr, n_estimators=n, max_depth=2, random_state=SEED).fit(X_train, y_train)
        zeilen.append({"learning_rate": lr, "n_estimators": n,
                       "R² Train": gb.score(X_train, y_train), "R² Test": gb.score(X_test, y_test)})
print(pd.DataFrame(zeilen).round(3))

for name, m in [("Linear", LinearRegression()), ("Random Forest", RandomForestRegressor(300, random_state=SEED, n_jobs=-1)),
                ("Gradient Boosting", GradientBoostingRegressor(learning_rate=0.05, n_estimators=200, max_depth=2, random_state=SEED))]:
    print(f"{name:18s} R² CV: {cross_val_score(m, X_train, y_train, cv=5, scoring='r2').mean():.3f}")

**Deutung:** Hohe Lernrate mit vielen Runden: Train fast perfekt, Test deutlich schlechter (Overfitting). Am besten auf dem Test ist eine kleine bis mittlere Lernrate mit passender Rundenzahl. Auf diesem kleinen, fast linearen Datensatz ist die lineare Regression in der Cross-Validation sogar am besten. Komplexer ist nicht automatisch besser, darum lohnt sich immer eine einfache Baseline.